In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# 1. Настройка устройства
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# 2. Загрузка данных из стандартного пакета (может быть долгой)
# transform = transforms.ToTensor()

# trainset = torchvision.datasets.CIFAR10(root='./data',
#                                         train=True, download=True,
#                                         transform=transform)
# trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)

# testset = torchvision.datasets.CIFAR10(root='./data', train=False,
#                                        download=True, transform=transform)
# testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)



In [ ]:
# Если не удается загрузить CIFAR-10 через torchvision, можно использовать альтернативный способ загрузки данных

from cs231n.data_utils import load_CIFAR10
cifar10_dir = 'cs231n/datasets/cifar-10-batches-py'

X_train, y_train, X_test, y_test = load_CIFAR10(cifar10_dir)
# create pytorch datasets
trainset = torch.utils.data.TensorDataset(torch.tensor(X_train).permute(0, 3, 1, 2).float() / 255.0, torch.tensor(y_train))
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)

testset = torch.utils.data.TensorDataset(torch.tensor(X_test).permute(0, 3, 1, 2).float() / 255.0, torch.tensor(y_test))
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)

In [ ]:

# 3. Архитектура Сверточного Автокодировщика
# Как влияет размер латентного пространства на качество изображений?
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super(ConvAutoencoder, self).__init__()
        self.latent_space = 512
        # ЭНКОДЕР: X -> Латентное пространство Z
        # Вход: 3 x 32 x 32
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),  # -> 16 x 16 x 16
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), # -> 32 x 8 x 8
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, self.latent_space) # Латентное пространство размерности 64
        )

        # ДЕКОДЕР: Латентное пространство Z -> X_hat
        self.decoder_linear = nn.Linear(self.latent_space, 32 * 8 * 8)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1,
                               output_padding=1), # -> 16 x 16 x 16
            nn.ReLU(),
            nn.ConvTranspose2d(16, 3, kernel_size=3, stride=2, padding=1,
                               output_padding=1),  # -> 3 x 32 x 32
            nn.Sigmoid() # Ограничивает пиксели в диапазоне [0, 1]
        )

    def forward(self, x):
        # Полный проход X -> Z -> X_hat
        latent = self.encoder(x)

        # Разворачиваем вектор обратно в карту признаков для сверток
        x_hat = self.decoder_linear(latent)
        x_hat = x_hat.view(-1, 32, 8, 8)
        x_hat = self.decoder(x_hat)
        return x_hat, latent

model = ConvAutoencoder().to(device)

# 4. Функция потерь и оптимизатор
# Сравниваем пиксели оригинала и восстановленной картинки (MSE)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 5. Обучение модели
epochs = 10
print("Начало обучения автокодировщика...")
for epoch in range(epochs):
    running_loss = 0.0
    for inputs, _ in trainloader: # Метки классов (_) нам не нужны для X -> X
        inputs = inputs.to(device)

        optimizer.zero_grad()
        outputs, _ = model(inputs)

        # Функция потерь: насколько сильно восстановленный X отличается от исходного X
        loss = criterion(outputs, inputs)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Эпоха [{epoch+1}/{epochs}], Потери (MSE): {running_loss/len(trainloader):.4f}')
print("Обучение завершено!")

In [ ]:
model.eval()
with torch.no_grad():
    # Берем один батч из тестовой выборки
    images, _ = next(iter(testloader))
    images = images.to(device)
    reconstructed, _ = model(images)

    # Переводим в numpy для отрисовки
    images = images.cpu().numpy()
    reconstructed = reconstructed.cpu().numpy()

# Рисуем оригинал vs восстановление
fig, axes = plt.subplots(nrows=2, ncols=8, figsize=(12, 4))
for i in range(8):
    # Оригиналы
    axes[0, i].imshow(np.transpose(images[i], (1, 2, 0)))
    axes[0, i].axis('off')
    if i == 0: axes[0, i].set_title("Оригинал")

    # Восстановленные
    axes[1, i].imshow(np.transpose(reconstructed[i], (1, 2, 0)))
    axes[1, i].axis('off')
    if i == 0: axes[1, i].set_title("Восстановление")
plt.show()


In [ ]:
test_iterator = iter(testloader)

In [ ]:
with torch.no_grad():
    # 1. Берем новый чистый батч тензоров PyTorch (не numpy!)
    images_tensor, _ = next(test_iterator)
    images_tensor = images_tensor.to(device)

    # 2. Выбираем два разных изображения для интерполяции (например, под индексом 1 и 3)
    img_A = images_tensor[1].unsqueeze(0) # Добавляем размерность батча (1, 3, 32, 32)
    img_B = images_tensor[3].unsqueeze(0)

    # 3. Извлекаем их латентные векторы
    latent_A = model.encoder(img_A)
    latent_B = model.encoder(img_B)

    # 4. Создаем 8 промежуточных шагов между А и Б
    fig, axes = plt.subplots(1, 8, figsize=(15, 2))
    for i, alpha in enumerate(np.linspace(0, 1, 8)):
        # Линейно смешиваем скрытые смыслы: Z = alpha * A + (1 - alpha) * B
        mixed_latent = alpha * latent_A + (1 - alpha) * latent_B

        # Декодируем смешанный вектор в картинку
        gen_img = model.decoder_linear(mixed_latent).view(-1, 32, 8, 8)
        gen_img = model.decoder(gen_img).cpu().squeeze(0).numpy()

        axes[i].imshow(np.transpose(gen_img, (1, 2, 0)))
        axes[i].axis('off')
        axes[i].set_title(f"a={alpha:.1f}", fontsize=9)

    plt.suptitle("Интерполяция в латентном пространстве (от Объекта Б к Объекту А)", y=1.15)
    plt.show()


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# 1. Выбираем 3 класса для визуализации
# CIFAR-10 классы: 0='airplane', 1='automobile', 5='dog'
target_classes = [0, 1, 5]
class_names = ['Airplane', 'Automobile', 'Dog']

all_latents = []
all_labels = []

model.eval()
with torch.no_grad():
    for inputs, labels in testloader:
        inputs = inputs.to(device)

        # Фильтруем батч: оставляем только нужные 3 класса
        # Маска определяет, какие элементы батча входят в наш список классов
        mask = torch.isin(labels, torch.tensor(target_classes))

        if not mask.any():
            continue

        filtered_inputs = inputs[mask]
        filtered_labels = labels[mask]

        # Прогоняем через модель и забираем латентные векторы
        _, latents = model(filtered_inputs)

        all_latents.append(latents.cpu().numpy())
        all_labels.append(filtered_labels.numpy())

# Объединяем результаты в массивы
all_latents = np.concatenate(all_latents, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

# 2. Настройка и запуск t-SNE (с исправленным max_iter)
tsne = TSNE(
    n_components=2,
    perplexity=40,       # Оптимально для выборки из 3 классов (~3000 картинок)
    max_iter=2000,       # Актуальный параметр вместо устаревшего n_iter
    init='pca',          # Стабильный старт через PCA
    random_state=42
)

print(f"Снижение размерности для {len(all_latents)} объектов...")
latents_2d = tsne.fit_transform(all_latents)

# 3. Визуализация результатов
plt.figure(figsize=(10, 8))

# Маппинг исходных меток классов (0, 1, 5) в индексы (0, 1, 2) для красивой палитры цвета
colors = ['#1f77b4', '#ff7f0e', '#2ca02c'] # Синий, Оранжевый, Зеленый

for i, class_idx in enumerate(target_classes):
    # Находим индексы точек, принадлежащих текущему классу
    indices = (all_labels == class_idx)

    plt.scatter(
        latents_2d[indices, 0],
        latents_2d[indices, 1],
        c=colors[i],
        label=class_names[i],
        alpha=0.7,
        s=25,
        edgecolors='none'
    )

plt.legend(loc="upper right", fontsize=12)
plt.title("Визуализация латентного пространства для 3 классов CIFAR-10", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()



In [ ]:
latent_dim = 1024

class VAE(nn.Module):
    def __init__(self, latent_dim):
        super(VAE, self).__init__()
        self.latent_dim = latent_dim

        # ЭНКОДЕР
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),  # -> 16 x 16 x 16
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), # -> 32 x 8 x 8
            nn.ReLU(),
            nn.Flatten()
        )

        # Вместо одного слоя эмбеддинга — два линейных слоя
        self.fc_mu = nn.Linear(32 * 8 * 8, latent_dim)       # Вектор средних
        self.fc_logvar = nn.Linear(32 * 8 * 8, latent_dim)   # Вектор лог-дисперсии

        # ДЕКОДЕР
        self.decoder_linear = nn.Linear(latent_dim, 32 * 8 * 8)
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 3, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar) # Получаем стандартное отклонение
        eps = torch.randn_like(std)   # Случайный шум с распределением N(0, I)
        return mu + eps * std         # Трюк репараметризации

    def forward(self, x):
        hidden = self.encoder_conv(x)
        mu = self.fc_mu(hidden)
        logvar = self.fc_logvar(hidden)

        z = self.reparameterize(mu, logvar)

        x_hat = self.decoder_linear(z).view(-1, 32, 8, 8)
        x_hat = self.decoder_conv(x_hat)
        return x_hat, mu, logvar

model = VAE(latent_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Специфичная функция потерь для VAE
def vae_loss_function(x_hat, x, mu, logvar):
    # 1. Ошибка реконструкции (MSE)
    recon_loss = nn.functional.mse_loss(x_hat, x, reduction='sum')

    # 2. Расхождение Кульбака-Лейблера (формула выведена аналитически для Гауссиан)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    return recon_loss + kl_loss

# Цикл обучения
model.train()
for epoch in range(10):
    total_loss = 0
    for inputs, _ in trainloader:
        inputs = inputs.to(device)
        optimizer.zero_grad()

        x_hat, mu, logvar = model(inputs)
        loss = vae_loss_function(x_hat, inputs, mu, logvar)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Эпоха {epoch+1}, Loss: {total_loss / len(trainset):.4f}")


In [ ]:
model.eval()
with torch.no_grad():
    # Создаем 8 абсолютно случайных векторов в латентном пространстве
    random_latent_vectors = torch.randn(8, latent_dim).to(device)

    # Пропускаем только через ДЕКОДЕР
    gen_images = model.decoder_linear(random_latent_vectors).view(-1, 32, 8, 8)
    gen_images = model.decoder_conv(gen_images).cpu().numpy()

# Отрисовка сгенерированных «галлюцинаций» модели
fig, axes = plt.subplots(1, 8, figsize=(15, 2))
for i in range(8):
    axes[i].imshow(np.transpose(gen_images[i], (1, 2, 0)))
    axes[i].axis('off')
plt.suptitle("Картинки, сгенерированные VAE из случайного шума", y=1.1)
plt.show()


# Задание на самостоятельную работу
1) Скопируйте код VAE, измените его и добейтесь лучшего результата генерации.  
Можно попробовать улучшить архитектуру, подобрать гиперпараметры или использовать более простой датасет, например, MNIST.   

2) Объясните, за счет чего удалось добиться улучшений. 

In [ ]:
# Copy code here!
